## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [8]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [9]:
OpenAIAPIKey = os.getenv('OPENAI_API_KEY')
GoogleAPIKey = os.getenv('GOOGLE_API_KEY')
GroqAPIKey = os.getenv('GROQ_API_KEY')

GeminiBaseURL = "https://generativelanguage.googleapis.com/v1beta/openai"
GroqBaseURL = "https://api.groq.com/openai/v1"

GeminiClient = AsyncOpenAI(base_url=GeminiBaseURL, api_key=GoogleAPIKey)
GroqClient = AsyncOpenAI(base_url=GroqBaseURL, api_key=GroqAPIKey)

GeminiModel = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=GeminiClient)
GroqModel = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=GroqClient)


In [10]:
Instructions = """
               You are a sales agent working for ComplAI, a company that provides a SaaS tool
               for ensuring SOC2 compliance and preparing for audits, powered by AI. You write
               compelling sales emails that are likely to get a response.
               """
SalesAgent1 = Agent(name="OpenAI Sales Agent", instructions=Instructions, model="gpt-5.4-nano")
SalesAgent2 = Agent(name="Gemini Sales Agent", instructions=Instructions, model=GeminiModel)
SalesAgent3 = Agent(name="Groq Sales Agent", instructions=Instructions, model=GroqModel)

ToolDesc = "Use this tool to write a cold sales email. Just instruct the tool to write a cold sales email"

Tool1 = SalesAgent1.as_tool(tool_name="SalesAgent1", tool_description=ToolDesc)
Tool2 = SalesAgent2.as_tool(tool_name="SalesAgent2", tool_description=ToolDesc)
Tool3 = SalesAgent3.as_tool(tool_name="SalesAgent3", tool_description=ToolDesc)

import sendgrid
from sendgrid.helpers.mail import Email, Mail, To, Content

@function_tool
def SendEmail(Subject: str, HTMLBody: str):
    """ Send an email to a prospective client """
    SG = sendgrid.SendGridAPIClient(api_key = os.getenv("SENDGRID_API_KEY"))
    FromEmail = Email("spraybubble@gmail.com")
    ToEmail = To("spraybubble@gmail.com")
    content = Content("text/html",HTMLBody)
    mail = Mail(FromEmail, ToEmail, Subject, content).get()
    SG.client.mail.send.post(request_body=mail)
    return {"Mail sent":"OK"}

Tools = [Tool1, Tool2, Tool3, SendEmail]


In [12]:
Instructions = """
               You are a Sales Manager at ComplAI. Your goal is to send the single best cold email to
               a prospect using your tools
               """

Task = """
       You have to send an email to a sales prospect. You are equipped with tools to write email
       drafts and a tool for sending the email. Please follow the below steps in order:

       1. Generate Drafts: Use each of the 3 SalesAgent tools to generate a draft email each. Do
       not proceed until you have the 3 email drafts
       2. Select Best Email: Use your judgement to decide which is the best of the 3 email drafts. 
       You must select one and only one final best email. Do not delegate judgement of the best
       email to any one the SalesAgent tools. You must make that decision yourself.
       3. Generate a subject line for the email. The subject line may be different from what the
       sales agents provide. Also, convert the body of the email to a well-formatted HTML body.
       4. Send the HTML body and the subject line to the SendEmail tool, which will send the email.

       Rules:
       - Do not write the email yourself. You must choose one of the 3 draft emails written by the
       sales agents
       - There should be ONE best email selected, not more than one.
       - You may choose a subject line different from what the sales agents provide
       - Do not stop until you have sent the email using the SendEmail tool.
       """

from agents import ModelSettings

SalesManager = Agent(name="Sales Manager", instructions=Instructions, tools = Tools,
                          model="gpt-5.4-mini",model_settings=ModelSettings(tool_choice="required"))

with trace("Using multiple models"):
    Result = await Runner.run(SalesManager,Task)
    print (Result.final_output)

Sent.


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [13]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [14]:
Email = """
        Hi [FirstName],

        I'm hitting you up to see if you'd like to buy our product. It's really great.
        You'll miss out if you don't buy it.

        Cheerio!

        Siddharth
        """

In [15]:
Checker = Agent(name="Email checker", instructions="You review potential sales emails",
                     model="gpt-5.4-mini", output_type=EmailReview)
Result = await Runner.run(Checker, Email)
Review = Result.final_output
import textwrap
print (textwrap.fill(str(Review), width=100))

print (Review.is_professional)

is_professional=False number_of_sentences=2 contains_placeholders=True
False


## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [18]:
@output_guardrail
async def GuardrailAgainstOutput(ctx, Agent, Message):
    Result = await Runner.run(Checker, Message, context=ctx.context)
    IsProblem = Result.final_output.contains_placeholders or not Result.final_output.is_professional
    return GuardrailFunctionOutput(output_info={"review":Result.final_output}, tripwire_triggered=IsProblem)

In [19]:
CowboyInstructions = Instructions + "\nSpeak like a cowboy"
SalesAgentCowboy = Agent(name="CowboyAgent", instructions=CowboyInstructions, model="gpt-5.4-nano",
                             output_guardrails=[GuardrailAgainstOutput])
Result = await Runner.run(SalesAgentCowboy, "Write a cold sales email")
print (Result.final_output)

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [21]:
SimpleCowboy = Agent(name="Simple Cowboy", instructions=CowboyInstructions, model=GeminiModel)
Result = await Runner.run(SimpleCowboy,"Write a cold sales email")
Email = Result.final_output
print ("Email generated: \n\n"+Email)

Review = await Runner.run(Checker, Email)
if Review.final_output.contains_placeholders or not Review.final_output.is_professional:
    print ("The email is not professional or has placeholders and will not be sent")
else:
    print ("Email is good")

Email generated: 

Subject: Y'all lookin’ to herd those AI compliance risks?

Howdy partner,

I’ve been keepin’ an eye on the trail you’re blaze-in’ over at [Prospect Company], and I reckon y’all are movin’ faster than a long-legged steer on a hot tin roof.

But here’s the hitch: when you’re movin’ that fast with AI, those pesky compliance hurdles have a nasty habit of sneakin’ up on your blind side like a rattlesnake in the tall grass. That’s where we come in at ComplAI. We’ve built the best fence in the business to keep your AI operations safe, sound, and square with the law—without pullin’ on the reins of your innovation.

I ain’t lookin’ to hogtie your time with a long-winded pitch. I just want to know if you’re interested in seein’ how we can keep your herd runnin’ smooth and avoidin’ any stampedes.

You got a spare ten minutes to chat later this week, or are you too busy wranglin’ calves?

Best regards,

[Your Name]
Sales Manager, ComplAI
The email is not professional or has plac

## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [ ]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [ ]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [ ]:
CODE_DIR

In [ ]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

In [ ]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [ ]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [ ]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

## OPTIONAL EXTRA: MCP Teaser!

In [23]:
from agents.mcp import MCPServerStreamableHttp


## Without an MCP Server

Task = """
       In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, 
       what is the role of the Manifest object?
       Be accurate and answer only if you know. Do not fabricate an answer.
       If you do not know the answer, say so.
       """

QuestionAgent = Agent (name="Question answerer", instructions="Answer the question posed to you", model="gpt-4o-mini")
Result = await Runner.run(QuestionAgent, Task)
print ("Answer without MCP: \n\n")
print (Result.final_output)

## With an MCP Server

Params = {"url":"https://mcp.context7.com/mcp", "timeout":60}

async with MCPServerStreamableHttp(name="Context7", params=Params) as Server:
    QuestionAgent = Agent(name="Question answerer", mcp_servers = [Server], model="gpt-4o-mini",
                               instructions = "Use Context7 to answer the question posed to you")
    Result = await Runner.run(QuestionAgent,Task)

print ("Answer with MCP: \n\n")
print (Result.final_output)

Answer without MCP: 


I don't have information about features introduced in May 2026, including the role of the Manifest object in the OpenAI Agents SDK. You may want to refer to the official documentation or announcements for the most accurate information.
Answer with MCP: 


In the OpenAI Agents SDK's SandboxAgents feature, the **Manifest** object serves a crucial role in defining the environment and capabilities for sandboxed agent execution. It encompasses various elements including:

1. **Entries**: These define accessible resources like directories and files within the sandbox, establishing what the agent can read or modify.
  
2. **User Configurations**: It allows for the specification of users and their permissions, which dictate what actions the agent can perform in relation to mounted and output paths.

3. **Security and Isolation**: By providing a structured way to isolate agent operations, the Manifest ensures that agents work within specified constraints, enhancing securi

And see the traces:

https://platform.openai.com/traces
